In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [6]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/anora7/ecg-processed-data/train_spectro.npz
/kaggle/input/datasets/anora7/ecg-processed-data/test_raw.npz
/kaggle/input/datasets/anora7/ecg-processed-data/test_spectro.npz
/kaggle/input/datasets/anora7/ecg-processed-data/train_raw.npz
/kaggle/input/datasets/anora7/ecg-processed-data/best_threshold.json
/kaggle/input/datasets/anora7/ecg-processed-data/best_model.pt
/kaggle/input/datasets/anora7/models/stage1_best_model.pt
/kaggle/input/datasets/anora7/models/stage2_info.json
/kaggle/input/datasets/anora7/models/stage2_best_model.pt


In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b2
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              classification_report, confusion_matrix,
                              matthews_corrcoef, roc_auc_score)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
 
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

PyTorch : 2.10.0+cu128
CUDA    : True


In [2]:
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_PATH   = "/kaggle/input/datasets/anora7/ecg-processed-data"
MODELS_PATH = "/kaggle/input/datasets/anora7/models"
OUT_DIR     = "/kaggle/working/phase3_results"
os.makedirs(OUT_DIR, exist_ok=True)
 
STAGE1_CKPT = os.path.join(MODELS_PATH, "stage1_best_model.pt")
STAGE2_CKPT = os.path.join(MODELS_PATH, "stage2_best_model.pt")
STAGE2_INFO = os.path.join(MODELS_PATH, "stage2_info.json")
THRESHOLD_F = os.path.join(DATA_PATH,   "best_threshold.json")
CLASS_NAMES = ["Normal (N)", "AF (A)", "Other (O)", "Noisy (~)"]
BATCH_SIZE  = 64
 
print("Device     :", DEVICE)
print("Output dir :", OUT_DIR)

Device     : cuda
Output dir : /kaggle/working/phase3_results


## Loaded phase-2 stage2 info 

In [3]:
with open(THRESHOLD_F) as f:
    STAGE1_THRESHOLD = json.load(f)["stage1_threshold"]
 
with open(STAGE2_INFO) as f:
    stage2_info = json.load(f)
 
print(f"Stage 1 threshold : {STAGE1_THRESHOLD}")
print(f"Stage 2 val F1s   : AF={stage2_info['val_f1_AF']:.4f}  "
      f"OT={stage2_info['val_f1_OT']:.4f}  NO={stage2_info['val_f1_NO']:.4f}")

Stage 1 threshold : 0.3
Stage 2 val F1s   : AF=0.9656  OT=0.9880  NO=0.9508


## Test Data

In [4]:
print("Loading test data...")
raw_data     = np.load(os.path.join(DATA_PATH, "test_raw.npz"),     allow_pickle=True)
spectro_data = np.load(os.path.join(DATA_PATH, "test_spectro.npz"), allow_pickle=True)
 
X_test_raw  = raw_data["windows"]          # (N, 4500)
y_test_4cls = raw_data["labels"]           # (N,)  0=N,1=A,2=O,3=~
X_test_spec = spectro_data["spectrograms"] # (M, 3, 224, 224) non-Normal only
y_test_s2   = spectro_data["labels"]       # (M,)  0=A,1=O,2=~
 
print(f"Test raw signals  : {X_test_raw.shape}")
print(f"Test spectrograms : {X_test_spec.shape}")
print("4-class label counts:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {(y_test_4cls == i).sum()}")

Loading test data...
Test raw signals  : (3496, 4500)
Test spectrograms : (1454, 3, 224, 224)
4-class label counts:
  Normal (N): 2042
  AF (A): 307
  Other (O): 1067
  Noisy (~): 80


## Stage 1 TCN Building blocks

In [5]:
# Two thin nn.Module wrappers are unavoidable — PyTorch requires forward()
# for the causal trim + residual logic and SE gating. Everything else is plain functions.
 
def make_se_gate(channels, reduction=16):
    return nn.Sequential(
        nn.AdaptiveAvgPool1d(1),
        nn.Flatten(),
        nn.Linear(channels, channels // reduction),
        nn.ReLU(),
        nn.Linear(channels // reduction, channels),
        nn.Sigmoid()
    )
 
def make_temporal_parts(in_ch, out_ch, kernel_size=9, dilation=1, dropout=0.2):
    pad   = (kernel_size - 1) * dilation
    conv1 = nn.utils.weight_norm(
                nn.Conv1d(in_ch,  out_ch, kernel_size, dilation=dilation, padding=pad))
    conv2 = nn.utils.weight_norm(
                nn.Conv1d(out_ch, out_ch, kernel_size, dilation=dilation, padding=pad))
    proj  = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    return conv1, conv2, nn.PReLU(), nn.PReLU(), nn.Dropout(dropout), nn.Dropout(dropout), proj
 
class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=9, dilation=1, dropout=0.2):
        super().__init__()
        (self.conv1, self.conv2, self.act1, self.act2,
         self.drop1, self.drop2, self.proj) = make_temporal_parts(
             in_ch, out_ch, kernel_size, dilation, dropout)
    def forward(self, x):
        out = self.drop1(self.act1(self.conv1(x)))
        out = self.drop2(self.act2(self.conv2(out)))
        out = out[:, :, :x.size(2)]
        return out + self.proj(x)
 
class SE1D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.gate = make_se_gate(channels, reduction)
    def forward(self, x):
        return x * self.gate(x).unsqueeze(-1)
 
def build_stage1_tcn(dropout=0.2):
    tcn  = nn.Sequential(
        TCNBlock(1,   64,  dilation=1, dropout=dropout),
        TCNBlock(64,  128, dilation=2, dropout=dropout),
        TCNBlock(128, 256, dilation=4, dropout=dropout),
    )
    se   = SE1D(256)
    pool = nn.AdaptiveAvgPool1d(1)
    head = nn.Sequential(
        nn.Linear(256, 64), nn.PReLU(), nn.Dropout(dropout), nn.Linear(64, 2)
    )
    model = nn.Sequential()
    model.add_module("tcn",  tcn)
    model.add_module("se",   se)
    model.add_module("pool", pool)
    model.add_module("head", head)
    return model
 
def forward_stage1(model, x):
    x   = x.unsqueeze(1)
    out = model.tcn(x)
    out = model.se(out)
    out = model.pool(out).squeeze(-1)
    return model.head(out)

## Stage 2 EfficientNet building block

In [6]:
def build_stage2_efficientnet(dropout=0.4):
    model = efficientnet_b2(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout, inplace=True),
        nn.Linear(in_features, 3)
    )
    return model
 
def forward_stage2(model, x):
    return model(x)

## Load weights 

In [7]:
stage1_model = build_stage1_tcn(dropout=0.2).to(DEVICE)
stage1_model.load_state_dict(torch.load(STAGE1_CKPT, map_location=DEVICE))
stage1_model.eval()
print("Stage 1 weights loaded")
 
stage2_model = build_stage2_efficientnet(dropout=0.4).to(DEVICE)
stage2_model.load_state_dict(torch.load(STAGE2_CKPT, map_location=DEVICE))
stage2_model.eval()
print("Stage 2 weights loaded ")

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Stage 1 weights loaded
Stage 2 weights loaded 


## Stage 1 inference

In [8]:
def run_stage1_inference(model, X, threshold):
    all_probs, all_preds = [], []
    with torch.no_grad():
        for start in range(0, len(X), BATCH_SIZE):
            x_batch = torch.tensor(
                X[start:start+BATCH_SIZE], dtype=torch.float32).to(DEVICE)
            with torch.amp.autocast("cuda"):
                logits = forward_stage1(model, x_batch)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend((probs[:, 1] >= threshold).astype(int))
    return np.array(all_preds), np.array(all_probs)
 
print(f"Running Stage 1 inference (threshold={STAGE1_THRESHOLD})...")
s1_preds, s1_probs = run_stage1_inference(stage1_model, X_test_raw, STAGE1_THRESHOLD)
 
s1_true = (y_test_4cls != 0).astype(int)
s1_f1   = f1_score(s1_true, s1_preds, average="macro")
s1_acc  = np.mean(s1_preds == s1_true)
print(f"Stage 1 binary accuracy : {s1_acc:.4f}")
print(f"Stage 1 binary macro F1 : {s1_f1:.4f}")
print(f"  Normal predicted     : {(s1_preds==0).sum()}")
print(f"  NonNormal predicted  : {(s1_preds==1).sum()}")

Running Stage 1 inference (threshold=0.3)...
Stage 1 binary accuracy : 0.8518
Stage 1 binary macro F1 : 0.8446
  Normal predicted     : 2210
  NonNormal predicted  : 1286


## Stage 2 inference

In [9]:
def run_stage2_inference(model, X):
    all_probs, all_preds = [], []
    with torch.no_grad():
        for start in range(0, len(X), BATCH_SIZE):
            x_batch = torch.tensor(
                X[start:start+BATCH_SIZE], dtype=torch.float32).to(DEVICE)
            with torch.amp.autocast("cuda"):
                logits = forward_stage2(model, x_batch)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(probs.argmax(axis=1))
    return np.array(all_preds), np.array(all_probs)
 
print("Running Stage 2 inference...")
s2_preds, s2_probs = run_stage2_inference(stage2_model, X_test_spec)
 
s2_f1  = f1_score(y_test_s2, s2_preds, average="macro")
s2_acc = np.mean(s2_preds == y_test_s2)
print(f"Stage 2 accuracy : {s2_acc:.4f}")
print(f"Stage 2 macro F1 : {s2_f1:.4f}")

Running Stage 2 inference...
Stage 2 accuracy : 0.8776
Stage 2 macro F1 : 0.7912


## Hierarchical combination

In [17]:
# Stage 2 label remap: 0→1 (AF), 1→2 (Other), 2→3 (Noisy)
S2_TO_4CLASS = {0: 1, 1: 2, 2: 3}

final_preds = np.zeros(len(y_test_4cls), dtype=int)
prob_4cls   = np.zeros((len(y_test_4cls), 4), dtype=np.float32)

# Get indices where Stage 1 predicts NonNormal
nonnormal_idx = np.where(s1_preds == 1)[0]

# Stage 2 has exactly as many predictions as test_spectro windows
# test_spectro was built from non-Normal TRUE windows in Phase 1
# So we need to align by true non-Normal positions, not Stage 1 predictions
true_nonnormal_idx = np.where(y_test_4cls != 0)[0]

# Assign Stage 1 Normal predictions first
for i in np.where(s1_preds == 0)[0]:
    final_preds[i]  = 0
    prob_4cls[i, 0] = s1_probs[i, 0]

# Assign Stage 2 predictions to true non-Normal positions
# (Stage 2 was run on test_spectro which contains exactly these windows in order)
for j, i in enumerate(true_nonnormal_idx):
    if j < len(s2_preds):
        if s1_preds[i] == 0:
            # Stage 1 missed this — predicted Normal, override with Stage 2
            final_preds[i]  = 0  # keep Stage 1 decision
            prob_4cls[i, 0] = s1_probs[i, 0]
        else:
            final_preds[i]  = S2_TO_4CLASS[s2_preds[j]]
            prob_4cls[i, 0] = s1_probs[i, 0]
            prob_4cls[i, 1] = s2_probs[j, 0] * s1_probs[i, 1]
            prob_4cls[i, 2] = s2_probs[j, 1] * s1_probs[i, 1]
            prob_4cls[i, 3] = s2_probs[j, 2] * s1_probs[i, 1]

print(f"True non-Normal windows  : {len(true_nonnormal_idx)}")
print(f"Stage 2 predictions      : {len(s2_preds)}")
print("Final prediction distribution:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {(final_preds==i).sum()}")

True non-Normal windows  : 1454
Stage 2 predictions      : 1454
Final prediction distribution:
  Normal (N): 2385
  AF (A): 277
  Other (O): 764
  Noisy (~): 70


In [18]:
# Normalise rows so they sum to 1 for AUC
row_sums = prob_4cls.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
prob_4cls = prob_4cls / row_sums

## Evaluation

In [19]:
print("\nClassification Report:")
print(classification_report(y_test_4cls, final_preds,
      target_names=CLASS_NAMES, digits=4))
 
macro_f1   = f1_score(y_test_4cls, final_preds, average="macro")
macro_prec = precision_score(y_test_4cls, final_preds, average="macro", zero_division=0)
macro_rec  = recall_score(y_test_4cls, final_preds, average="macro", zero_division=0)
mcc        = matthews_corrcoef(y_test_4cls, final_preds)
acc        = np.mean(final_preds == y_test_4cls)
 
f1_n  = f1_score(y_test_4cls, final_preds, labels=[0], average="macro")
f1_af = f1_score(y_test_4cls, final_preds, labels=[1], average="macro")
f1_ot = f1_score(y_test_4cls, final_preds, labels=[2], average="macro")
f1_no = f1_score(y_test_4cls, final_preds, labels=[3], average="macro")
 
print(f"Overall accuracy  : {acc:.4f}")
print(f"Macro F1          : {macro_f1:.4f}")
print(f"Macro Precision   : {macro_prec:.4f}")
print(f"Macro Recall      : {macro_rec:.4f}")
print(f"MCC               : {mcc:.4f}")
print(f"\nPer-class F1:")
print(f"  Normal F1 : {f1_n:.4f}  {'PASS ' if f1_n  >= 0.88 else 'FAIL'} (target ≥ 0.88)")
print(f"  AF     F1 : {f1_af:.4f}  {'PASS ' if f1_af >= 0.80 else 'FAIL'} (target ≥ 0.80)")
print(f"  Other  F1 : {f1_ot:.4f}  {'PASS ' if f1_ot >= 0.70 else 'FAIL'} (target ≥ 0.70)")
print(f"  Noisy  F1 : {f1_no:.4f}  {'PASS ' if f1_no >= 0.80 else 'FAIL'} (target ≥ 0.80)")
print(f"\nPlan A combined target (macro F1 ≥ 0.88): "
      f"{'PASS ' if macro_f1 >= 0.88 else 'FAIL'} ({macro_f1:.4f})")
 
try:
    auc = roc_auc_score(y_test_4cls, prob_4cls, multi_class="ovr", average="macro")
    print(f"Macro AUC (OvR)   : {auc:.4f}")
except Exception as e:
    print(f"AUC could not be computed: {e}")


Classification Report:
              precision    recall  f1-score   support

  Normal (N)     0.8562    1.0000    0.9225      2042
      AF (A)     0.8123    0.7329    0.7705       307
   Other (O)     0.9045    0.6476    0.7548      1067
   Noisy (~)     0.6857    0.6000    0.6400        80

    accuracy                         0.8598      3496
   macro avg     0.8147    0.7451    0.7720      3496
weighted avg     0.8632    0.8598    0.8515      3496

Overall accuracy  : 0.8598
Macro F1          : 0.7720
Macro Precision   : 0.8147
Macro Recall      : 0.7451
MCC               : 0.7485

Per-class F1:
  Normal F1 : 0.9225  PASS  (target ≥ 0.88)
  AF     F1 : 0.7705  FAIL (target ≥ 0.80)
  Other  F1 : 0.7548  PASS  (target ≥ 0.70)
  Noisy  F1 : 0.6400  FAIL (target ≥ 0.80)

Plan A combined target (macro F1 ≥ 0.88): FAIL (0.7720)
AUC could not be computed: Target scores need to be probabilities for multiclass roc_auc, i.e. they should sum up to 1.0 over classes


## Stage 1 error analysis

In [20]:
s1_cm = confusion_matrix(s1_true, s1_preds)
print("\nStage 1 error analysis:")
print(f"  True Normal predicted as NonNormal (false alarm) : {s1_cm[0,1]}")
print(f"  True NonNormal predicted as Normal (miss)        : {s1_cm[1,0]}")
print(f"  Total Stage 1 errors                             : {s1_cm[0,1] + s1_cm[1,0]}")


Stage 1 error analysis:
  True Normal predicted as NonNormal (false alarm) : 175
  True NonNormal predicted as Normal (miss)        : 343
  Total Stage 1 errors                             : 518


## Confusion matrix

In [21]:
cm  = confusion_matrix(y_test_4cls, final_preds)
fig, ax = plt.subplots(figsize=(7, 6))
im  = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(CLASS_NAMES, rotation=15)
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Hierarchical Classifier — Confusion Matrix (Test Set)")
for i in range(4):
    for j in range(4):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=100)
plt.show()
print("Saved: confusion_matrix.png")

Saved: confusion_matrix.png


In [22]:
summary = {
    "macro_f1":         float(macro_f1),
    "macro_precision":  float(macro_prec),
    "macro_recall":     float(macro_rec),
    "mcc":              float(mcc),
    "accuracy":         float(acc),
    "f1_Normal":        float(f1_n),
    "f1_AF":            float(f1_af),
    "f1_Other":         float(f1_ot),
    "f1_Noisy":         float(f1_no),
    "stage1_binary_f1": float(s1_f1),
    "stage2_macro_f1":  float(s2_f1),
    "stage1_threshold": float(STAGE1_THRESHOLD),
}
with open(os.path.join(OUT_DIR, "phase3_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("Saved: phase3_summary.json")
 
print("\n" + "="*60)
print("PHASE 3 COMPLETE")
print("="*60)
print(f"Macro F1  : {macro_f1:.4f}")
print(f"MCC       : {mcc:.4f}")
print(f"Accuracy  : {acc:.4f}")
print("\nDownload from /kaggle/working/phase3_results/:")
print("  confusion_matrix.png")
print("  phase3_summary.json")

Saved: phase3_summary.json

PHASE 3 COMPLETE
Macro F1  : 0.7720
MCC       : 0.7485
Accuracy  : 0.8598

Download from /kaggle/working/phase3_results/:
  confusion_matrix.png
  phase3_summary.json
